In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
file_path="Titanic_Dataset.csv"

#Reading csv file
df=pd.read_csv(file_path)

In [42]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,0,-0.557460,1,0,-0.502445,2,0,0,0,1,0,0
1,1,1,1,0.649091,1,0,0.786845,2,0,0,0,0,1,0
2,1,3,1,-0.255822,0,0,-0.488854,1,1,0,1,0,0,0
3,1,1,1,0.422862,1,0,0.420730,2,0,0,0,0,1,0
4,0,3,0,0.422862,0,0,-0.486337,1,1,0,0,1,0,0


In [43]:
class TitleExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["Title"] = X["Name"].str.extract(r' ([A-Za-z]+)\.')

        X["Title"] = X["Title"].replace({
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs"
        })

        rare_titles = ["Dr", "Rev", "Major", "Col", "Don", "Lady",
                        "Sir", "Capt", "Countess", "Jonkheer"]
        X["Title"] = X["Title"].replace(rare_titles, "Rare")

        return X

In [44]:
class AgeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ages_ = X.groupby("Title")["Age"].median()
        return self

    def transform(self, X):
        X = X.copy()
        for title, median_age in self.median_ages_.items():
            mask = (X["Age"].isna()) & (X["Title"] == title)
            X.loc[mask, "Age"] = median_age

        X["Age"] = X["Age"].fillna(self.median_ages_.median())
        return X

In [45]:
class FamilyFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
        return X

In [46]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

In [47]:
encode_scale = ColumnTransformer(
    transformers=[
        ("sex_encode", OrdinalEncoder(categories=[["male", "female"]]), ["Sex"]),
        ("title_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Title"]),
        ("scale_numeric", StandardScaler(), ["Age", "Fare"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
).set_output(transform="pandas")

NameError: name 'OrdinalEncoder' is not defined

In [48]:
pipe = Pipeline([
    ("extract_title", TitleExtractor()),
    ("impute_age", AgeImputer()),
    ("family_features", FamilyFeatures()),
    ("drop_columns", DropColumns(columns=["Name", "PassengerId", "Ticket"])),
    ("encode_scale", encode_scale),
    ("model", LogisticRegression(max_iter=1000))
])

NameError: name 'encode_scale' is not defined

In [49]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

NameError: name 'train_test_split' is not defined

In [ ]:
pipe.fit(X_train, y_train)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Title'],
      dtype='str')

In [ ]:
y_pred = pipe.predict(X_test)


Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64